# Create dataset using preprocessed data

In [1]:
import csv

with open("processed-actual-outages.csv") as file:
    csv_reader = csv.DictReader(file)
    actual_data = list(csv_reader)

In [2]:
from datetime import datetime

from tqdm import tqdm

# convert dates to datetime objects, and PTID and Voltage to integers
date_keys = ["OutDatetime", "MinTimeStamp", "MaxTimeStamp"]

for row in tqdm(actual_data):
    row["PTID"] = int(row["PTID"])
    row["Voltage"] = int(row["Voltage"])
    for key in date_keys:
        row[key] = datetime.strptime(row[key], "%Y-%m-%d %H:%M:%S")

actual_data[0]

100%|██████████| 74063/74063 [00:00<00:00, 111748.34it/s]


{'PTID': 26053,
 'Name': 'MOUNTAIN-SWANROAD_115_104-3',
 'OutDatetime': datetime.datetime(2005, 1, 1, 0, 0),
 'MinTimeStamp': datetime.datetime(2012, 9, 11, 11, 52, 17),
 'MaxTimeStamp': datetime.datetime(2013, 8, 8, 8, 37, 17),
 'Voltage': 115,
 'FirstBus': 'MOUNTAIN',
 'SecondBus': 'SWANROAD',
 'OutageType': 'Planned'}

In [3]:
from pathlib import Path

output_path = Path("output")
output_path.mkdir(exist_ok=True)

# HYPER PARAMETERS

In [4]:
EVENT_WINDOW_HOURS = 6
MIN_YEAR = 2008

VOLTAGES = [69, 115, 132, 220, 345, 500, 735]
VOLTAGE_GROUP = {
    # 69 or less
    27: 69,
    34: 69,
    69: 69,
    # 155/120
    115: 115,
    120: 115,
    # 132/138
    132: 132,
    138: 132,
    # 220/230
    220: 220,
    230: 220,
    # 345
    345: 345,
    #500
    500: 500,
    # 735/765
    735: 735,
    765: 735,
}

INTERVALS_MINUTES = [15, 30, 60, 120, 240, 480]


In [5]:
actual_data = sorted([row for row in actual_data if row["MinTimeStamp"].year >= MIN_YEAR and row["OutDatetime"].year >= MIN_YEAR], key=lambda x: x["MinTimeStamp"])

In [10]:
# load graph
import igraph

g = igraph.Graph.Read_Pickle("res/outage_graph.pkl")


In [11]:
# load buses/nodes information
import json

bus_name_to_index_path = Path("res/bus_name_to_index.json")
bus_name_to_index = json.loads(bus_name_to_index_path.read_text())

In [7]:
from datetime import timedelta

dataset = []

delta_time = timedelta(hours=EVENT_WINDOW_HOURS)
for i, ref_row in enumerate(actual_data):
    if ref_row["MinTimeStamp"].year < MIN_YEAR or ref_row["OutDatetime"].year < MIN_YEAR:
        continue

    event_window_start = row["MinTimeStamp"] - delta_time
    window = [row for row in actual_data[:i] if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < delta_time]
    if len(window) > 3:
        break

len(window)

4

In [8]:
window

[{'PTID': 25566,
  'Name': 'ASTORIAW-ASTORIA5_138_24125M',
  'OutDatetime': datetime.datetime(2008, 1, 2, 4, 12),
  'MinTimeStamp': datetime.datetime(2008, 1, 2, 4, 17, 18),
  'MaxTimeStamp': datetime.datetime(2008, 2, 2, 23, 57, 18),
  'Voltage': 138,
  'FirstBus': 'ASTORIAW',
  'SecondBus': 'ASTORIA5',
  'OutageType': 'Planned'},
 {'PTID': 25565,
  'Name': 'ASTORIAW-ASTORIA5_138_24125L',
  'OutDatetime': datetime.datetime(2008, 1, 2, 4, 13),
  'MinTimeStamp': datetime.datetime(2008, 1, 2, 4, 17, 18),
  'MaxTimeStamp': datetime.datetime(2008, 2, 2, 23, 57, 18),
  'Voltage': 138,
  'FirstBus': 'ASTORIAW',
  'SecondBus': 'ASTORIA5',
  'OutageType': 'Planned'},
 {'PTID': 25304,
  'Name': 'EGRDNCTY-NEWBRDGE_138_463',
  'OutDatetime': datetime.datetime(2008, 1, 2, 6, 10),
  'MinTimeStamp': datetime.datetime(2008, 1, 2, 6, 12, 18),
  'MaxTimeStamp': datetime.datetime(2008, 1, 3, 23, 57, 18),
  'Voltage': 138,
  'FirstBus': 'EGRDNCTY',
  'SecondBus': 'NEWBRDGE',
  'OutageType': 'Planned'},
 

In [ ]:
import numpy as np



def extract_features(ref_row, window):
    ############
    # features #
    ############
    # simple counting features
    num_events = len(window)
    num_unique_ptids = len(set(row["PTID"] for row in window))
    voltage_group_list = {i: 0 for i in VOLTAGES}
    for row in window:
        voltage_group_list[VOLTAGE_GROUP[row["Voltage"]]] += 1
    voltage_group_list = [voltage_group_list[v] for v in VOLTAGES]
    num_planned = len([row for row in window if row["OutageType"] == "Planned"])
    num_auto = len([row for row in window if row["OutageType"] == "Auto"])
    bus_names = [row[bus] for row in window for bus in ["FirstBus", "SecondBus"]]
    num_unique_buses = len(set(bus_names))

    # fine-grained interval features
    fine_interval_features = []
    for dt in INTERVALS_MINUTES:
        dt = timedelta(minutes=dt)
        num_events_interval = len([row for row in window if ref_row["MinTimeStamp"] - row["MinTimeStamp"] < dt])
        fine_interval_features.append(num_events_interval)

    # graph based features
    node_degrees = np.array([g.degree(bus_name_to_index[bus_name]) for bus_name in bus_names])
    node_degrees_mean = node_degrees.mean().item()
    node_degrees_std = node_degrees.std().item()
    node_degrees_min = node_degrees.min().item()
    node_degrees_max = node_degrees.max().item()
    node_degrees_stats = [node_degrees_mean, node_degrees_std, node_degrees_min, node_degrees_max]

    ##########
    # labels #
    ##########
    # Auto/Planned labels
    is_auto = True if ref_row["OutageType"] == "Auto" else False

    # time to reference event
    time_to_event = ref_row["MinTimeStamp"] - max(row["MinTimeStamp"] for row in window)

    return num_events, num_unique_ptids, voltage_group_list, num_planned, num_auto, num_unique_buses, fine_interval_features, is_auto, time_to_event.total_seconds(), node_degrees_stats


extract_features(ref_row, window)

(4,
 4,
 [0, 0, 4, 0, 0, 0, 0],
 4,
 0,
 6,
 [0, 0, 0, 0, 2, 4],
 False,
 9000.0,
 [6.375, 3.6721077053920954, 2, 11])